# 第 8 讲 实验 — 检索增强生成（RAG）
## 从任正非讲话语料，看清 RAG 的每一步

第 8 讲讲的是 **RAG（Retrieval-Augmented Generation，检索增强生成）**：不改动模型参数，而是在提问时**先检索**一小段相关证据、再让模型基于证据回答。本实验把这条流水线拆开，让你亲手跑一遍：

```text
语料  →  分块  →  建索引  →  检索  →  证据审计  →  引用来源的 grounded answer
```

在本实验中你将：
1. **审计一个知识库**——405 篇任正非 1994–2019 年的公开讲话/访谈。
2. 用**零依赖的 TF-IDF 检索**把问题匹配到最相关的讲话片段。
3. 生成一个**带 `[年份 · 标题]` 引用**、可逐句审计的离线回答。
4. **动手实现**一个最小检索器，理解 TF-IDF 到底在做什么。
5. （可选）对比**语义 embedding 检索**与 **naive LLM vs RAG** 的差别。

> **前置条件：** 先完成 `Lec01_02_Lab_Getting_Started.ipynb`，它用清华镜像装好了 Python 环境。本实验的**默认路径只用 Python 标准库**——不需要 GPU、不需要 API key、不需要联网、不需要额外安装任何包。若某个可选环节缺包，用清华镜像安装，例如：
> `pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers`

*一句话直觉：* RAG 不是让模型"更聪明"，而是**改变模型能看到的证据**，并把答案**约束**在可核查的语料上。

In [ ]:
# 环境自检（Quick environment check）——只用标准库
import sys
from pathlib import Path
from collections import Counter

# 让 notebook 能 import 同文件夹下的 rag_ren.py / questions.py
ROOT = Path.cwd()
if not (ROOT / "rag_ren.py").exists():
    raise RuntimeError("请从 Lec08_RAG_Lab 文件夹启动本 notebook（该文件夹里有 rag_ren.py）。")
sys.path.insert(0, str(ROOT))

import rag_ren
from questions import QUESTIONS

CORPUS_DIR = rag_ren.DEFAULT_CORPUS_DIR
BUILD_DIR = ROOT / "build"
BUILD_DIR.mkdir(exist_ok=True)

print("Python  :", sys.version.split()[0])
print("语料目录:", CORPUS_DIR)
print("语料存在:", CORPUS_DIR.is_dir())
if not CORPUS_DIR.is_dir():
    print("\n⚠️  没找到语料。三种解决办法（任选其一）：")
    print("   1) 保证你有完整课程仓库（语料在 MiniCourse_8hr/demos/Ren-master/）；")
    print("   2) 把 Ren-master 复制到本文件夹下并改名为 corpus/；")
    print("   3) 设置环境变量 REN_CORPUS_DIR 指向语料目录。")
else:
    print("✅ 环境就绪，可以开始第 8 讲实验")

## 第 1 步：语料审计——先问"我们给了系统什么知识？"

RAG 的第一步不是调用模型，而是搞清楚知识库到底是什么。运行下面的 cell，看看语料的规模和年份跨度。

In [ ]:
docs = rag_ren.load_corpus(CORPUS_DIR)          # [(相对路径, 年份, 标题, 正文), ...]
year_counts = Counter(year for _rel, year, _title, _text in docs)
char_count = sum(len(text) for _rel, _year, _title, text in docs)

print(f"文档数: {len(docs)}")
print(f"字符数: {char_count:,}")
print(f"年份  : {min(year_counts)}–{max(year_counts)}（共 {len(year_counts)} 年）")

# 看一篇样本：标题 + 前 120 字
rel, year, title, text = docs[len(docs)//2]
print(f"\n样本讲话 → [{year} · {title}]")
print("  ", text[:120].replace(chr(10), " "), "...")

## 第 2 步：分块 + 建立 TF-IDF 索引

一篇讲话动辄几千字，直接塞给模型既超长又稀释重点。所以要**分块（chunking）**：切成约 500 字、带 80 字重叠的小段，每段保留**年份和标题**以便日后引用。

这里我们**故意用透明的 TF-IDF 稀疏检索**，而不是黑箱 embedding API——先看清 RAG 的机制，第 8 步再讨论生产系统为什么会换成向量数据库。

> **TF-IDF 一句话：** 一个词在**本段**出现越多（TF↑）、在**整个语料**里越罕见（IDF↑），它对这段的代表性就越强。查询和每个 chunk 都变成"词→权重"的向量，按**余弦相似度**排序。

In [ ]:
CHUNK_SIZE = 500      # 每块字符数
OVERLAP    = 80       # 相邻块的重叠字符数（防止把一句话从中间切断）

index, chunks, docs = rag_ren.build_index(
    corpus_dir=CORPUS_DIR, backend="tfidf",
    chunk_size=CHUNK_SIZE, overlap=OVERLAP, verbose=False,
)
summary = rag_ren.corpus_summary(docs, chunks)
print(f"文档数     : {summary['documents']}")
print(f"chunk 数   : {summary['chunks']}")
print(f"TF-IDF 词表: {len(index.idf):,} 个 token")
print(f"chunk_size = {CHUNK_SIZE} | overlap = {OVERLAP}")

> **✏️ 你来试试：** 把上面的 `CHUNK_SIZE` 改成 `200`（太碎）和 `1500`（太粗），各重跑一次这两个 cell，观察 chunk 数怎么变。想一想：块太小 / 太大，分别会怎样影响后面检索到的证据质量？

## 第 3 步：十个讨论问题

这些**不是**事实查询，而是商学院会争论的战略判断题。每题都配了：常规看法（conventional view）、任正非在语料中的真实立场（ren view），以及若干 **anchor**（预期应命中的讲话文件名片段），用来审计检索是否抓对了材料。

In [ ]:
try:
    from IPython.display import Markdown, display
    rows = ["| # | 主题 | 问题 |", "|---|------|------|"]
    for it in QUESTIONS:
        q = it["q"]; short = q[:50] + ("…" if len(q) > 50 else "")
        rows.append(f"| {it['id']} | {it['theme']} | {short} |")
    display(Markdown("\n".join(rows)))
except Exception:
    for it in QUESTIONS:
        print(f"{it['id']:>2}. [{it['theme']}] {it['q']}")

## 第 4 步：先审计检索质量（在任何生成之前）

好的工程习惯：**先看检索、再谈生成**。下面对每道题跑 anchor 诊断——只要 top-6 里命中了至少一个预期 anchor，就说明检索没跑偏。注意这里**一次模型都没调用**。

In [ ]:
diagnostics = rag_ren.retrieval_diagnostics(index, QUESTIONS, top_k=6)
for row in diagnostics:
    status = "PASS" if row["anchor_hit_count"] >= 1 else "FAIL"
    print(f"{status} | Q{row['id']:02d} | top_score={row['top_score']:.3f} | "
          f"anchors={row['anchor_hit_count']}/{row['anchor_count']} | {row['theme']}")
all_ok = all(r["anchor_hit_count"] >= 1 and r["top_score"] > 0.01 for r in diagnostics)
print(f"\n所有检索检查通过: {all_ok}")

## 第 5 步：单题走查——看看检索到的"证据"长什么样

改 `QUESTION_ID` 换题。先**只看 top chunks，不生成答案**：请你像审稿人一样判断——这些材料，够不够支撑一个有依据的回答？

In [ ]:
QUESTION_ID = 4        # 4 = 灰度 / 决策哲学；可改成 1..10
TOP_K = 6

item = next(it for it in QUESTIONS if it["id"] == QUESTION_ID)
question = item["q"]
retrieved = index.search(question, top_k=TOP_K)

print(f"Q{QUESTION_ID}【{item['theme']}】\n{question}\n")
print("检索到的 top chunks：\n")
for rank, r in enumerate(retrieved, 1):
    c = r.chunk
    preview = " ".join(c.text.split())[:240]
    print(f"{rank}. score={r.score:.3f}  [{c.year} · {c.title}]")
    print(f"   来源: {c.source_file}")
    print(f"   {preview}…\n")

hits = rag_ren.hits_anchor(retrieved, item.get("anchors", []), top_n=TOP_K)
print(f"anchor 命中: {len(hits)}/{len(item.get('anchors', []))}  {hits}")

> **✏️ 你来试试：** 在下面的 cell 里写一个**你自己的问题**（中文即可），看看检索抓回什么。试一个语料里"应该有"的话题（如"研发投入""上市""天才少年"），再试一个"应该没有"的话题（如"元宇宙估值"），比较两者的 `top_score`。

In [ ]:
MY_QUESTION = "华为为什么坚持不上市？"     # ← 改成你想问的问题

for rank, r in enumerate(index.search(MY_QUESTION, top_k=5), 1):
    c = r.chunk
    print(f"{rank}. score={r.score:.3f}  [{c.year} · {c.title}]")
    print(f"   {' '.join(c.text.split())[:160]}…")

## 第 6 步：生成"引用来源"的 grounded answer（离线，无 LLM）

这一步用**抽取式**方法：从检索到的 chunk 里挑最相关的句子，并给每句附上 `[年份 · 标题]` 引用。它不如大模型流畅，但**每句话都能追溯到证据**——这正是 RAG 的核心承诺：**引用是答案契约的一部分，不是装饰**。

先看**喂给模型的提示词（prompt）**是怎么拼出来的：指令 + 检索到的上下文 + 用户问题。

In [ ]:
prompt = rag_ren.build_prompt(question, retrieved)
print(prompt[:700], "\n…（省略）…\n")

print("="*60)
result = rag_ren.compare(question, index, top_k=TOP_K, llm_backend="extractive")
print("问题:", result["question"])
print("\n抽取式 grounded answer（每条都带引用）:\n")
print(result["rag"])

## 第 7 步：✏️ 动手实现一个最小检索器

TF-IDF 听起来高级，核心其实很朴素：**把问题和每个 chunk 都切成词，数一数它们共享了多少个词，共享得多就排前面。** 下面请你实现这个"婴儿版"检索器 `overlap_search`，再和 `index`（真正的 TF-IDF）比一比 top-1。

你会用到 `rag_ren.tokenize(text)`——把中英文切成 token（中文按字 + 二元组）。**先自己写，参考答案在最后的附录。**

In [ ]:
def overlap_search(query, chunks, top_k=6):
    """按"与问题共享的 token 数"给每个 chunk 打分，返回 top_k 个 (score, chunk)。
    步骤提示：
      1. q_tokens = set(rag_ren.tokenize(query))
      2. 对每个 chunk：c_tokens = set(rag_ren.tokenize(chunk.text))
      3. score = 两个集合交集的大小  ->  len(q_tokens & c_tokens)
      4. 收集 (score, chunk)，按 score 从大到小排序
      5. 返回前 top_k 个
    （参考答案见附录。）
    """
    # your code here
    raise NotImplementedError("请实现 overlap_search —— 参考答案在附录。")


# —— 实现后运行这里，和真正的 TF-IDF 对比 ——
try:
    mine = overlap_search(question, chunks, top_k=3)
    print("你的 overlap_search top-1 :", rag_ren.format_citation(mine[0][1]))
    print("TF-IDF        top-1      :", rag_ren.format_citation(index.search(question, top_k=1)[0].chunk))
    print("\n讨论：两者一样吗？如果不同，想想 TF-IDF 的 IDF 权重是怎么压低'我们/公司/发展'这类高频词的。")
except NotImplementedError as e:
    print("尚未实现：", e)

## 第 8 步（可选）：语义 embedding 检索——当关键词对不上时

TF-IDF 的软肋是**必须字面命中**。若你问"灰度管理"，而语料写的是"在黑白之间保留妥协与宽容"，两者没有共同的词，TF-IDF 就抓不到。

**Embedding（嵌入）** 把任意文本映射成一个稠密向量 $E:\text{text}\to\mathbb{R}^d$，让**意思相近的文本，向量也相近**——即使一个字都不重叠。

> ⚠️ **这一节是可选的，且需要联网下载约 120 MB 模型**（首次运行）。在中国大陆能否下载取决于网络；下不动**不影响**前面任何步骤。安装：`pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers`

In [ ]:
import time
EMBED_OK = False
try:
    embed_index = rag_ren.LocalEmbeddingIndex(
        chunks=chunks,
        embedding_model=rag_ren.DEFAULT_LOCAL_EMBED_MODEL,
        cache_path=BUILD_DIR / "local_embedding_cache.json",
    )
    t0 = time.time()
    embed_index.build(verbose=True)          # 首次下载模型 + 编码；之后从缓存秒开
    print(f"\n[embed] 就绪，用时 {time.time()-t0:.1f}s；向量维度 = {len(embed_index.chunk_vectors[0])}")
    EMBED_OK = True
except Exception as exc:
    print(f"[embed] 跳过（{type(exc).__name__}: {exc}）")
    print('若想启用：pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers，然后重跑本 cell。')

In [ ]:
# 对同一个问题，比较 TF-IDF 与 embedding 的 top-1；只有装好 embedding 才运行。
if EMBED_OK:
    tf1 = index.search(question, top_k=1)[0]
    em1 = embed_index.search(question, top_k=1)[0]
    print(f"问题: {question}\n")
    print(f"TF-IDF   top-1: score={tf1.score:.3f}  {rag_ren.format_citation(tf1.chunk)}")
    print(f"Embedding top-1: score={em1.score:.3f}  {rag_ren.format_citation(em1.chunk)}")
    print("\n注意：换检索后端时，build_prompt / 抽取式回答 / 引用格式都没变——只换了'检索'这一层。")
else:
    print("跳过——先在上一格装好并构建 embedding 索引。")

## 第 9 步（可选）：naive LLM vs RAG——把差别看给学生

默认**不运行**。若课堂上要演示"无检索的裸模型"和"有检索的 RAG"回答差异，把 `ENABLE_LLM` 改成 `True`。这会调用本机 `claude -p`，需要你已经在终端运行过 `claude` 并 `/login`。

In [ ]:
ENABLE_LLM = False        # 只有在终端跑过 `claude` 且 /login 后，才改成 True
LLM_MODEL = "sonnet"

if ENABLE_LLM:
    ok, msg = rag_ren.claude_code_auth_check(timeout=30)
    print(msg)
    if not ok:
        raise RuntimeError("Claude Code 未就绪：请在终端运行 `claude` 并 /login。")
    res = rag_ren.compare(question, index, top_k=TOP_K,
                          llm_backend="claude-code", llm_model=LLM_MODEL)
    print("\n----- Naive（无检索）-----\n", res["naive"])
    print("\n----- RAG（有检索上下文）-----\n", res["rag"])
else:
    print("已跳过。把 ENABLE_LLM = True 可运行可选的 Claude Code 对比。")

## 课堂讨论 / 小结

建议按这个顺序讨论：
1. 先让同学**凭直觉**给出常规答案（conventional view）。
2. 展示 **top retrieved chunks**，让同学判断证据是否相关、是否够用。
3. 展示 **抽取式 grounded answer**，强调**引用契约**：每句都能追溯到年份与讲话。
4. （可选）运行 Claude Code，对比 naive 与 RAG 的回答。
5. 追问：如果把检索器换成 embedding，RAG 架构里**哪一层变了、哪一层没变**？

**关键结论：** RAG = *检索* + *生成*。它不改模型权重，而是**改变可用证据**，并把回答**约束到可审计的语料**上。想深入，请看第 8 讲 T2（分块与 embedding）和 T3（向量检索 / GraphRAG）。

### Make it your own
- 换一份你自己的语料（一批 `.md` 文件放进一个文件夹，用 `REN_CORPUS_DIR` 指过去），重跑全流程。
- 给 `questions.py` 加一道你关心的战略题，并写出它的 anchor。
- 把 `overlap_search` 升级：给高频词降权（IDF 思想），看排序有没有更接近 TF-IDF。

---
## 附录 · 参考答案

先自己写第 7 步，再看这里。

In [ ]:
# --- 参考答案 Solution: overlap_search ---
def overlap_search(query, chunks, top_k=6):
    q_tokens = set(rag_ren.tokenize(query))
    scored = []
    for c in chunks:
        c_tokens = set(rag_ren.tokenize(c.text))
        score = len(q_tokens & c_tokens)          # 共享 token 数
        scored.append((score, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

mine = overlap_search(question, chunks, top_k=3)
print("overlap_search top-3:")
for score, c in mine:
    print(f"  overlap={score:>3}  {rag_ren.format_citation(c)}")
print("\nTF-IDF top-1:", rag_ren.format_citation(index.search(question, top_k=1)[0].chunk))
print("\n为什么可能不同：overlap 只数'共享几个词'，'我们/公司/发展'这类高频词会灌水；"
      "TF-IDF 用 IDF 给它们降权，所以更看重'灰度/妥协'这类有区分度的词。")